# SortFlow: Waste Image Classification (MLOps Summative)

## 1. Problem statement and imports

**Problem.** Sorting waste at the source (household, market, or office) is slow and error-prone. SortFlow is an image classifier that takes a photo of a single waste item and predicts which of six material classes it belongs to, so it can be sorted into the right bin.

**Classes (6):** `cardboard`, `glass`, `metal`, `organic`, `paper`, `plastic`

**Why this fits an MLOps project.** Waste looks different from place to place (local packaging, lighting, backgrounds). A model trained on public data will drift when it meets local photos, so the system must let users upload new labelled images and retrain the model without touching the code.

**What this notebook covers**

| Section | Content |
|---|---|
| 2 | Data acquisition, class merging, train/val/test split |
| 3 | Exploratory data analysis (3 interpreted features) |
| 4 | Preprocessing and augmentation (`src/preprocessing.py`) |
| 5 | Baseline CNN vs. optimized custom CNN, evaluation, saving the model |
| 6 | Results, error analysis, limitations |

**Reusable code lives in `src/`** (`preprocessing.py`, `model.py`, `prediction.py`), so the notebook, the retraining pipeline, and the API all use exactly the same preprocessing.

**Environment:** Kaggle notebook with a GPU. The TensorFlow version printed at the end of this notebook is the same one pinned in the Dockerfile.

# 2. Data loading and class distribution (bar chart)

**Data source.** [Garbage Classification (Kaggle, mostafaabla)](https://www.kaggle.com/datasets/mostafaabla/garbage-classification), licensed ODbL-1.0. It has 12 folders. We keep the ones that map to our six classes and drop `battery`, `clothes`, `shoes` and `trash`, which are outside the project scope.

**Class mapping:** the three glass folders (`brown-glass`, `green-glass`, `white-glass`) are merged into `glass`, and `biological` is renamed `organic`.

**Split:** a per-class random split of **70% train / 15% validation / 15% test** with a fixed seed (42), so every class appears in every split. The test set is never used for training or for choosing hyperparameters.

The first cell below lists the input folder so the dataset path can be checked. The second cell does the merge and split into `/kaggle/working/waste_data`.

In [1]:
!find /kaggle/input -maxdepth 5 -type d

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/mostafaabla
/kaggle/input/datasets/mostafaabla/garbage-classification
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/battery
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/biological
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/brown-glass
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/cardboard
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/clothes
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/green-glass
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/metal
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification/paper
/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_cl

In [2]:
import random, shutil
from pathlib import Path

random.seed(42)
SRC = Path('/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification')  # replace with the path from Step 1
DST = Path('/kaggle/working/waste_data')

assert SRC.exists(), f'SRC not found: {SRC}'

MAPPING = {
    'cardboard': 'cardboard', 'metal': 'metal', 'paper': 'paper',
    'plastic': 'plastic', 'biological': 'organic',
    'brown-glass': 'glass', 'green-glass': 'glass', 'white-glass': 'glass',
}

by_class = {}
for src_cls, tgt in MAPPING.items():
    for f in (SRC / src_cls).glob('*'):
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            by_class.setdefault(tgt, []).append(f)

assert by_class, 'No images found. Check SRC and the class folder names.'

for cls, files_ in by_class.items():
    random.shuffle(files_)
    n = len(files_)
    splits = {'train': files_[:int(.7*n)],
              'val':   files_[int(.7*n):int(.85*n)],
              'test':  files_[int(.85*n):]}
    for split, items in splits.items():
        out = DST / split / cls
        out.mkdir(parents=True, exist_ok=True)
        for f in items:
            shutil.copy(f, out / f"{f.parent.name}_{f.name}")
    print(cls, n)

cardboard 891
metal 769
paper 1050
plastic 865
organic 985
glass 2011


In [4]:
import random, shutil
from pathlib import Path

DST = Path('/kaggle/working/waste_data')
assert DST.exists(), 'Run the split cell first (the one that creates waste_data)'
CLASSES = sorted(p.name for p in (DST/'train').iterdir())

random.seed(1)
OUT = Path('/kaggle/working/sample_data')
for split, n in [('train', 30), ('test', 20)]:
    for cls in CLASSES:
        files = random.sample(list((DST/split/cls).glob('*')), n)
        out = OUT/split/cls
        out.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy(f, out/f.name)

shutil.make_archive('/kaggle/working/sample_data', 'zip', OUT)
print(CLASSES)

['cardboard', 'glass', 'metal', 'organic', 'paper', 'plastic']


# 3. Exploratory data analysis

Three features of the dataset are analysed below, and each one leads to a design decision:

1. **Class distribution** → decides which metrics to trust (per-class F1, not accuracy alone).
2. **Image dimensions and aspect ratio** → decides the resize strategy.
3. **Brightness and color per class** → predicts which classes the model will confuse.

First, the number of images per class in each split.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

DST = Path('/kaggle/working/waste_data')
CLASSES = sorted(p.name for p in (DST/'train').iterdir())

rows = []
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        rows.append((split, cls, len(list((DST/split/cls).glob('*')))))
df_counts = pd.DataFrame(rows, columns=['split', 'class', 'count'])
print(df_counts.pivot(index='class', columns='split', values='count'))

## Feature 1: Class distribution (imbalance)

In [ ]:
train_counts = df_counts[df_counts.split == 'train'].set_index('class')['count']
ax = train_counts.plot(kind='bar', figsize=(8, 4), color='steelblue')
ax.set_title('Training images per class'); ax.set_ylabel('Images')
for i, v in enumerate(train_counts): ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout(); plt.show()
print('Imbalance ratio (max/min):', round(train_counts.max() / train_counts.min(), 2))

**Interpretation**

- `glass` is by far the largest class (because three folders were merged into it), and `metal` is the smallest. The largest class is about **2.6x** the size of the smallest.
- This is a *moderate* imbalance. A model that leans toward `glass` would still look decent on overall accuracy, so accuracy alone can hide poor performance on `metal` or `plastic`.
- **Decision:** report per-class precision, recall and F1 plus a confusion matrix, and use **macro F1** (every class counts equally) as the main summary metric. Inverse-frequency class weights are available in `compute_class_weights()`. They were tested, but the final model trains without them (see the experiments table in section 6).

## Visual check: sample images per class

Five random training images per class. Look for classes that are visually hard to tell apart (for example paper vs. cardboard, or glass vs. plastic bottles) and for how varied the backgrounds are.

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(12, 2.4 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    files = random.sample(list((DST/'train'/cls).glob('*')), 5)
    for c, f in enumerate(files):
        axes[r, c].imshow(Image.open(f).convert('RGB')); axes[r, c].axis('off')
        if c == 0: axes[r, c].set_title(cls, loc='left', fontsize=11)
plt.tight_layout(); plt.show()

**Interpretation**

- Backgrounds, lighting and camera distance vary a lot inside a single class, which is good for generalization but makes the task harder.
- Transparent items (glass, clear plastic) pick up the color of their surroundings, so shape and texture matter more than color for these classes.
- Note for the limitations section: this dataset contains augmented and near-duplicate images, so test accuracy is probably a little optimistic compared to photos taken in the real world.

## Feature 2: Image dimensions and aspect ratio

In [ ]:
recs = []
for cls in CLASSES:
    for f in random.sample(list((DST/'train'/cls).glob('*')), 100):
        w, h = Image.open(f).size
        recs.append((cls, w, h, w / h))
df_size = pd.DataFrame(recs, columns=['class', 'w', 'h', 'aspect'])
print(df_size.groupby('class')[['w', 'h', 'aspect']].agg(['mean', 'std']).round(2))
df_size.boxplot(column='aspect', by='class', figsize=(8, 4)); plt.suptitle(''); plt.show()

**Interpretation**

> **Fill in from your output above:** the typical width and height, how much they vary, and the aspect-ratio range per class.

- The images do not all share one size or shape, but a CNN needs a fixed input shape, so every image is resized to **224 x 224** in `load_dataset()` (training) and in `preprocess_image()` (the API).
- Using the same function for both training and prediction means the model sees identically prepared inputs in production.
- Resizing to a square slightly distorts non-square images. That is accepted because the distortion is mild and applies equally in training and prediction.

## Feature 3: Brightness and color per class

In [ ]:
recs = []
for cls in CLASSES:
    for f in random.sample(list((DST/'train'/cls).glob('*')), 100):
        a = np.asarray(Image.open(f).convert('RGB').resize((64, 64)), dtype=np.float32)
        recs.append((cls, a.mean(), *a.mean(axis=(0, 1))))
df_col = pd.DataFrame(recs, columns=['class', 'brightness', 'R', 'G', 'B'])
print(df_col.groupby('class').mean().round(1))
df_col.boxplot(column='brightness', by='class', figsize=(8, 4)); plt.suptitle(''); plt.show()
df_col.groupby('class')[['R', 'G', 'B']].mean().plot(kind='bar', color=['r', 'g', 'b'], figsize=(8, 4))
plt.title('Mean RGB per class'); plt.show()

**Interpretation**

> **Fill in from your output above:** which classes have the highest and lowest mean brightness, and which classes have similar mean RGB.

- Classes with similar color profiles (typically `cardboard` and `paper`, both brown or beige) are expected to be confused with each other. This is a prediction we check against the confusion matrix in section 5.
- Large brightness spread inside a class means lighting varies, so brightness and contrast augmentation is useful.

## EDA summary

| Finding | Consequence for the project |
|---|---|
| Glass about 2.6x larger than metal | Use macro F1 and per-class metrics, not accuracy alone |
| Mixed image sizes and shapes | Resize everything to 224 x 224 with one shared function |
| Overlapping color profiles and varied lighting | Add brightness, contrast, flip, rotation and zoom augmentation |

# 4. Preprocessing: resize to 224x224, rescale inside the model, and augmentation

`src/preprocessing.py` provides:

- `load_dataset(directory, shuffle, augment, class_names)`: loads a folder-per-class directory into a batched `tf.data` pipeline, resized to 224 x 224 with one-hot labels. Passing `class_names` forces the same label order as the original model, which matters when retraining on a folder that is missing a class.
- `augmentation`: random horizontal flip, rotation (15%), zoom (15%), brightness (20%) and contrast (20%). Applied **only to training data**.
- `compute_class_weights(directory)`: inverse-frequency weights for the imbalance.
- `preprocess_image(image)`: turns a single uploaded image into a model-ready batch for the API.

**Pixel scaling (0-255 to 0-1) happens inside the model** with a `Rescaling(1/255)` layer. The API and the retraining code therefore only need to resize, and scaling can never drift between training and prediction.

The repository is cloned so the notebook can import the exact same modules the API uses.

In [ ]:
import sys, random, shutil, json
from pathlib import Path

!git clone https://github.com/ShamiPlacide/SortFlow.git /kaggle/working/waste-classifier
sys.path.insert(0, '/kaggle/working/waste-classifier')

import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
from src import preprocessing
from src.preprocessing import load_dataset, compute_class_weights

Quick check that the pipeline works: the class order (alphabetical, which is also saved to `class_names.json` for the API) and the class weights.

In [ ]:
train_ds, class_names = load_dataset(DST/'train', shuffle=True, augment=True)
val_ds, _ = load_dataset(DST/'val', class_names=class_names)
print(class_names)
print(compute_class_weights(DST/'train'))

# 5. Model, evaluation, and saving

Two models are trained and compared on the same data. The comparison shows that the optimizations in the custom model are what make it work, not just a bigger network.

| | Baseline CNN | Custom CNN (final) |
|---|---|---|
| Architecture | 2 conv layers, flatten, dense 64 | 4 blocks (32, 64, 128, 256 filters) with 2 convs each |
| Regularization | none | BatchNorm, dropout (0.1 per block, 0.3 before output), L2 (1e-5) |
| Head | flatten + dense | global average pooling + dense 256 |
| Augmentation | none | flip, rotation, zoom, brightness, contrast |
| Optimizer | Adam, lr 1e-3 | Adam, lr 1e-3 with ReduceLROnPlateau |
| Early stopping | no | yes (patience 10, restores best weights) |

Both models are trained from scratch. No pretrained weights are used, so the saved model can be retrained directly as our own pre-trained model.

### 5.1 Baseline CNN

A deliberately simple network with no augmentation and no regularization. It is expected to overfit, and it gives a reference point for the custom model.

In [ ]:
# Step 2: Baseline model
import tensorflow as tf
from src.model import build_baseline, build_custom
from src.preprocessing import load_dataset, compute_class_weights

train_ds, class_names = load_dataset(DST/'train', shuffle=True, augment=False)
val_ds, _ = load_dataset(DST/'val', class_names=class_names)
test_ds, _ = load_dataset(DST/'test', class_names=class_names)

baseline = build_baseline(len(class_names))
hist_base = baseline.fit(train_ds, validation_data=val_ds, epochs=10)

### 5.2 Custom optimized CNN

Optimization techniques used: **data augmentation, batch normalization, dropout, L2 regularization, Adam optimizer, learning-rate scheduling (`ReduceLROnPlateau`), early stopping with best-weights restore, and checkpointing** of the best epoch.

Cell 1 below prepares the augmented training set and the callbacks. Cell 2 pulls the latest `src/` code. Cell 3 builds and trains the model for up to 60 epochs.

In [ ]:
# Step 3: Custom optimized model
train_aug, _ = load_dataset(DST/'train', shuffle=True, augment=True, class_names=class_names)
class_weights = compute_class_weights(DST/'train')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint('/kaggle/working/best_custom.keras',
                                       monitor='val_loss', save_best_only=True),
]

In [ ]:
!cd /kaggle/working/waste-classifier && git pull

In [ ]:
from src.model import build_custom

callbacks[0] = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model = build_custom(len(class_names))
hist = model.fit(train_aug, validation_data=val_ds, epochs=60, callbacks=callbacks)
# model = build_custom(len(class_names))
# hist = model.fit(train_aug, validation_data=val_ds, epochs=40,
                 # class_weight=class_weights, callbacks=callbacks)

### 5.3 Save the model

The model is saved as `.h5`, and the class names are saved next to it so the API always maps outputs to the right labels.

In [ ]:
import json
model.save('/kaggle/working/waste_cnn.h5')
json.dump(class_names, open('/kaggle/working/class_names.json', 'w'))

### 5.4 Training curves

Left: training (dashed) and validation accuracy. Right: validation loss. For the baseline, look for the gap between training and validation accuracy and for validation loss rising again. For the custom model, look for validation tracking training closely.

In [ ]:
# Step 4: Training curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, h in [('baseline', hist_base), ('custom', hist)]:
    ax[0].plot(h.history['val_accuracy'], label=f'{name} val')
    ax[0].plot(h.history['accuracy'], '--', label=f'{name} train')
    ax[1].plot(h.history['val_loss'], label=f'{name} val')
ax[0].set_title('Accuracy'); ax[1].set_title('Validation loss')
ax[0].legend(); ax[1].legend(); plt.show()

### 5.5 Evaluation on the held-out test set

The test set is used once, after training. Metrics reported: **loss, accuracy, precision, recall, F1 (per class and macro), and the confusion matrix.**

> The Keras `precision` and `recall` printed in the first line use a fixed 0.5 probability cutoff, so uncertain predictions count as misses. The scikit-learn report (argmax prediction) is the main result.

In [ ]:
# Step 5: Evaluation on the test set
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate(m, name):
    y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds])
    y_prob = m.predict(test_ds, verbose=0)
    y_pred = y_prob.argmax(1)
    loss, acc, prec, rec = m.evaluate(test_ds, verbose=0)
    print(f'\n== {name} ==\nloss={loss:.3f} acc={acc:.3f} precision={prec:.3f} recall={rec:.3f}')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(xticks_rotation=45)
    plt.title(name); plt.show()

evaluate(baseline, 'Baseline CNN')
evaluate(model, 'Custom CNN')

# 6. Results, error analysis and limitations

## Final test results (988 test images)

| Model | Loss | Accuracy | Macro precision | Macro recall | Macro F1 |
|---|---|---|---|---|---|
| Baseline CNN | 2.199 | 0.593 | 0.571 | 0.573 | 0.568 |
| **Custom CNN** | **0.542** | **0.817** | **0.819** | **0.799** | **0.805** |

**Per-class F1**

| Class | Baseline | Custom |
|---|---|---|
| cardboard | 0.714 | 0.876 |
| glass | 0.646 | 0.831 |
| metal | 0.326 | 0.667 |
| organic | 0.682 | 0.918 |
| paper | 0.573 | 0.810 |
| plastic | 0.467 | 0.728 |

*(Numbers are from the final run. Update them if you retrain and re-run the notebook.)*

## What the results show

- **The optimizations matter.** The custom CNN improves accuracy from 59.3% to 81.7% and cuts test loss from 2.20 to 0.54. The baseline's high test loss shows it memorizes the training data and is overconfident on new images.
- **`organic` is easiest** (F1 0.918), most likely because food waste has distinctive texture and color. **`metal` is hardest** (F1 0.667, recall 0.578), partly because it is the smallest class and partly because metal items vary a lot in shape and reflectivity.
- **Macro F1 (0.805) is close to accuracy (0.817)**, so the model is not just riding on the large `glass` class.

> **Fill in from your confusion matrix:** which class `metal` is most often predicted as, and whether `cardboard` and `paper` are confused with each other as the EDA predicted.

## Experiments that led to the final model

| Version | Changes | Accuracy | Macro F1 | Notes |
|---|---|---|---|---|
| Custom v1 | 1 conv per block, dropout 0.2 per block, L2 1e-4, class weights on | 0.628 | 0.630 | Underfit. `glass` recall only 0.43. Heavy regularization plus low class weight for `glass` made the model avoid predicting it |
| **Custom v2 (final)** | 2 convs per block, dropout 0.1 per block and 0.3 at the head, L2 1e-5, no class weights, early-stopping patience 10 | **0.817** | **0.805** | `glass` recall rose to 0.86. The imbalance (about 2.6x) was too mild for class weights to help |

## Retraining

The saved `waste_cnn.h5` is reused as the pre-trained model. `src/retrain.py` loads it, fine-tunes it with a low learning rate (1e-4) on newly uploaded images mixed with the original training data (to avoid forgetting), evaluates on the test set, and saves a new version. This notebook only produces the first model. The upload, preprocessing and retraining flow runs in the deployed application.

## Limitations

- The dataset contains augmented and near-duplicate images, so test accuracy is probably optimistic. Photos taken locally are the honest test of generalization.
- The model only knows six classes. Any other object (for example a flag) is forced into one of them. The UI should show a low-confidence warning.
- Single item per photo is assumed. Cluttered scenes with many items are out of scope.
- Training used a single random split, with no cross-validation.

## TensorFlow version

The version below is the one the model was saved with. The Dockerfile must pin the same version so the `.h5` loads identically in production.

In [ ]:
print('epochs run:', len(hist.history['loss']))
print('train acc:', hist.history['accuracy'][-1], 'val acc:', hist.history['val_accuracy'][-1])

In [ ]:
print(tf.__version__)